# Tarea 2 · Parte 2 — Evaluación de modelos y extracción estructurada

**Pregunta guía:** ¿Podemos convertir documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

Este notebook mide cuánto se equivocan los modelos (CER en documentos, WER en audio) sobre **10 documentos y 10 clips del
corpus de la Parte 1**, con transcripciones de referencia **hechas a mano**; compara una línea base con dos alternativas por
modalidad; extrae campos estructurados con un LLM y los audita a mano.

**Reglas que se respetan:**
- La muestra de evaluación se fija con una regla escrita *antes* de correr cualquier modelo (sección 1).
- Todos los modelos se evalúan sobre **los mismos** 10 documentos y 10 clips.
- Ninguna configuración se ajusta mirando estas métricas: las configuraciones están escritas abajo antes de medir.
- Referencias y salidas se normalizan con el mismo criterio (`voz.normalizar`: minúsculas, sin puntuación, con tildes);
  en WER se eliminan las marcas `[HABLANTE_n]`.

In [1]:
import os, json, re, time, subprocess, warnings
from datetime import date
from pathlib import Path

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import soundfile as sf
import jiwer

import herramientas as H
import voz
import llm

DATOS = Path("datos"); REF = DATOS / "referencias"; REF.mkdir(exist_ok=True)
RAW_DOC = DATOS / "raw" / "documentos"
docs = pd.read_csv(DATOS / "documentos.csv").fillna({"texto_ocr": "", "texto_referencia": ""})
clips = pd.read_csv(DATOS / "transcripciones.csv").fillna({"texto_asr": "", "texto_referencia": ""})
print(docs.shape, clips.shape)

(25, 7) (97, 11)


## 1 · Muestra de evaluación (fijada antes de medir)

**Documentos (10, los tres tipos):** se eligieron los de texto lo bastante corto para transcribirlo a mano completo,
cubriendo dificultades distintas:

| id | tipo | por qué |
|---|---|---|
| doc_001 | impreso | resolución municipal moderna, tipografía limpia sobre fondo decorado |
| doc_004 | impreso | portada de 1825, tipografía antigua y sello encima |
| doc_009 | foto | placa metálica inclinada, letras en relieve |
| doc_010 | foto | pizarra escrita con tiza |
| doc_011 | foto | cartel pintado a mano, letras decorativas |
| doc_013 | foto | letrero al aire libre, letra pequeña |
| doc_015 | foto | placa de mármol con poco contraste |
| doc_017 | manuscrito | formulario de 1895 llenado a mano |
| doc_018 | manuscrito | cursiva de Ricardo Palma (1912) |
| doc_022 | manuscrito | encabezados de Guaman Poma (c. 1615) |

**Clips (10, distintas fuentes y condiciones):** el **primer clip** (id más bajo) de cada una de estas fuentes, más los dos
primeros clips de OpenSLR 73. La regla depende solo del orden de los archivos, no de los resultados.
(La grabación de Chuyas, fuente_12, se había previsto pero su único clip se descartó en la Parte 1 por tener menos de 10 % de voz; se reemplazó por la otra entrevista de Cusco antes de calcular cualquier métrica.)

| fuente | condición |
|---|---|
| fuente_02 | entrevista en la calle, 2 hablantes (El Agustino) |
| fuente_04 | dirigente en una protesta, ruido de multitud |
| fuente_06 | entrevista en la calle (Magdalena) |
| fuente_09 | adulto mayor (Cusco) |
| fuente_10 | vendedora (Cusco) |
| fuente_14 | entrevista en un evento (Arequipa) |
| fuente_17 | noticiario de los años 40: narración con música |
| fuente_19 | noticiario antiguo: narración con música |
| openslr73 ×2 | habla leída en estudio |

In [2]:
EVAL_DOCS = ["doc_001", "doc_004", "doc_009", "doc_010", "doc_011", "doc_013", "doc_015", "doc_017", "doc_018", "doc_022"]
FUENTES_EVAL = ["fuente_02", "fuente_04", "fuente_06", "fuente_09", "fuente_10", "fuente_14", "fuente_17", "fuente_19"]

primeros = clips[clips["fuente_id"].isin(FUENTES_EVAL)].sort_values("id").groupby("fuente_id").head(1)
leidos = clips[clips["fuente_id"] == "openslr73"].sort_values("id").head(2)
EVAL_CLIPS = sorted(pd.concat([primeros, leidos])["id"])

faltan = set(EVAL_DOCS) - set(docs["id"])
assert not faltan, f"Estos documentos se descartaron en la Parte 1: {faltan}"
assert len(EVAL_CLIPS) == 10, EVAL_CLIPS

ev_docs = docs.set_index("id").loc[EVAL_DOCS].reset_index()
ev_clips = clips.set_index("id").loc[EVAL_CLIPS].reset_index()

# Plantillas para la transcripción manual: un .txt por caso en datos/referencias/
for i in EVAL_DOCS + EVAL_CLIPS:
    (REF / f"{i}.txt").touch(exist_ok=True)
display(ev_docs[["id", "tipo", "archivo"]])
display(ev_clips[["id", "fuente_id", "tipo_audio", "duracion_seg", "archivo_audio"]])

,id,tipo,archivo
0,doc_001,impreso,datos/raw/documentos/doc_001.jpg
1,doc_004,impreso,datos/raw/documentos/doc_004.jpg
2,doc_009,foto,datos/raw/documentos/doc_009.jpg
3,doc_010,foto,datos/raw/documentos/doc_010.jpg
4,doc_011,foto,datos/raw/documentos/doc_011.jpg
5,doc_013,foto,datos/raw/documentos/doc_013.jpg
6,doc_015,foto,datos/raw/documentos/doc_015.jpg
7,doc_017,manuscrito,datos/raw/documentos/doc_017.jpg
8,doc_018,manuscrito,datos/raw/documentos/doc_018_recorte.png
9,doc_022,manuscrito,datos/raw/documentos/doc_022.png


,id,fuente_id,tipo_audio,duracion_seg,archivo_audio
0,clip_004,fuente_02,entrevista,47.9,datos/raw/audio/clips/fuente_02_00.wav
1,clip_013,fuente_04,entrevista,45.8,datos/raw/audio/clips/fuente_04_00.wav
2,clip_024,fuente_06,entrevista,48.0,datos/raw/audio/clips/fuente_06_00.wav
3,clip_030,fuente_09,entrevista,48.5,datos/raw/audio/clips/fuente_09_00.wav
4,clip_036,fuente_10,entrevista,51.0,datos/raw/audio/clips/fuente_10_00.wav
5,clip_042,fuente_14,entrevista,46.2,datos/raw/audio/clips/fuente_14_00.wav
6,clip_052,fuente_17,noticiario,47.2,datos/raw/audio/clips/fuente_17_00.wav
7,clip_056,fuente_19,noticiario,48.9,datos/raw/audio/clips/fuente_19_00.wav
8,clip_058,openslr73,habla leída,11.8,datos/raw/audio/es_peru/pef_08421_00908604673.wav
9,clip_059,openslr73,habla leída,5.6,datos/raw/audio/es_peru/pef_01523_01958041774.wav


### Transcripción manual de referencia

Para cada caso se escribió a mano en `datos/referencias/<id>.txt`:

- **Documentos:** todo el texto visible, en orden de lectura (de arriba hacia abajo, de izquierda a derecha), respetando la
  ortografía original (p. ej. «Piru», «mas» sin tilde en textos antiguos). Las palabras ilegibles para una persona no se escriben.
- **Audio:** todo lo que se dice, escuchando el clip completo, sin marcas de hablante, sin muletillas repetidas («eh», «este»)
  ni ruidos. La música y la letra de canciones no se transcriben.

Las referencias **no** se copiaron de la salida de los modelos.

Nota: la referencia de clip_059 se había transcrito escuchando por error otro audio de OpenSLR; se detectó porque los tres modelos obtenían WER = 100 %. Se volvió a transcribir escuchando el archivo correcto. No se cambió ninguna configuración de modelo.

In [3]:
# Mismo diccionario de seudónimos de la Parte 1 (archivo privado, fuera de Git). Se aplica a las referencias
# y a todas las salidas de los modelos, para no publicar nombres de personas privadas y comparar en igualdad.
ruta_reemplazos = DATOS / "raw" / "reemplazos.json"
REEMPLAZOS = json.loads(ruta_reemplazos.read_text(encoding="utf-8")) if ruta_reemplazos.exists() else {}
anon = lambda t: H.seudonimizar_texto(t or "", REEMPLAZOS)[0]
print(f"Seudónimos cargados: {len(REEMPLAZOS)}")

def leer_ref(i):   # los .txt originales quedan fuera de Git; al CSV va la versión seudonimizada
    return anon((REF / f"{i}.txt").read_text(encoding="utf-8").strip())

ev_docs["texto_referencia"] = ev_docs["id"].map(leer_ref)
ev_clips["texto_referencia"] = ev_clips["id"].map(leer_ref)
vacias = [i for i in EVAL_DOCS + EVAL_CLIPS if not leer_ref(i)]
assert not vacias, f"Faltan referencias manuales: {vacias}"

# Se guardan en los CSV del corpus (vacío en el resto de filas)
docs["texto_referencia"] = docs["id"].map(dict(zip(ev_docs["id"], ev_docs["texto_referencia"]))).fillna("")
clips["texto_referencia"] = clips["id"].map(dict(zip(ev_clips["id"], ev_clips["texto_referencia"]))).fillna("")
docs.to_csv(DATOS / "documentos.csv", index=False)
clips.to_csv(DATOS / "transcripciones.csv", index=False)
print("Palabras de referencia:", ev_docs["texto_referencia"].str.split().str.len().sum(), "(documentos) ·",
      ev_clips["texto_referencia"].str.split().str.len().sum(), "(audio)")

Seudónimos cargados: 9
Palabras de referencia: 510 (documentos) · 819 (audio)


## 2 · Métricas

CER y WER **agregados** (errores totales / caracteres o palabras totales de referencia), que es lo que calcula `jiwer`
sobre la lista de casos; además se reporta el valor por caso para ver dónde falla cada modelo.

In [4]:
SIN_HABLANTE = re.compile(r"\[HABLANTE_\d+\]")

def limpiar_asr(t):
    return SIN_HABLANTE.sub(" ", t or "")

def cer_caso(ref, hip):
    return jiwer.cer(voz.normalizar(ref), voz.normalizar(hip) or "∅")

def wer_caso(ref, hip):
    return jiwer.wer(voz.normalizar(ref), voz.normalizar(limpiar_asr(hip)) or "∅")

resultados_docs, resultados_asr = [], []   # una fila por (modelo, caso)

## 3 · Documentos: línea base y alternativas

| Modelo | Configuración (fijada antes de medir) |
|---|---|
| **PaddleOCR base** | `PaddleOCR(lang="es")` con todas las opciones por defecto |
| **PaddleOCR Parte 1** | la configuración de la Parte 1: orientación por línea, sin desdoblado, imagen a 1600 px |
| **Tesseract 5** | `tesseract <img> stdout -l spa --psm 3` (instalado con `brew install tesseract tesseract-lang`) |

Todos leen la **misma** imagen preprocesada de la Parte 1 (`datos/raw/documentos/prep/`, solo rotación EXIF y tamaño).

In [5]:
import paddleocr
from paddleocr import PaddleOCR

def correr_ocr(nombre, leer):
    for d in ev_docs.itertuples():
        img = RAW_DOC / "prep" / f"{d.id}.jpg"
        t0 = time.perf_counter()
        texto = leer(img)
        seg = time.perf_counter() - t0
        texto = anon(texto)
        resultados_docs.append({"modelo": nombre, "id": d.id, "tipo": d.tipo, "texto": texto,
                                "cer": cer_caso(d.texto_referencia, texto), "segundos": seg})

base = PaddleOCR(lang="es")
base.predict(str(RAW_DOC / "prep" / f"{EVAL_DOCS[0]}.jpg"))           # calentamiento: no se cronometra
correr_ocr(f"PaddleOCR {paddleocr.__version__} base", lambda p: H.unir_lineas(H.leer_ocr(base, p)))

parte1 = H.crear_ocr()
parte1.predict(str(RAW_DOC / "prep" / f"{EVAL_DOCS[0]}.jpg"))
correr_ocr(f"PaddleOCR {paddleocr.__version__} Parte 1", lambda p: H.unir_lineas(H.leer_ocr(parte1, p)))

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/valeria/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/valeria/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/valeria/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/valeria/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually

In [6]:
import shutil
# Tesseract instalado con conda (o brew): se busca en el PATH y en las rutas habituales de Anaconda
TESS = shutil.which("tesseract") or next((str(p) for p in [Path.home() / "anaconda3/bin/tesseract",
        Path.home() / "opt/anaconda3/bin/tesseract", Path.home() / "miniconda3/bin/tesseract",
        Path("/opt/homebrew/bin/tesseract")] if p.exists()), None)
assert TESS, "No se encontró tesseract: conda install -c conda-forge tesseract"

def tesseract(ruta):
    r = subprocess.run([TESS, str(ruta), "stdout", "-l", "spa", "--psm", "3"],
                       capture_output=True, text=True, check=True)
    return r.stdout

version_tess = subprocess.run([TESS, "--version"], capture_output=True, text=True).stdout.split("\n")[0]
print(version_tess)
correr_ocr(f"Tesseract {version_tess.split()[-1]} (spa, psm 3)", tesseract)

rd = pd.DataFrame(resultados_docs)
tabla_cer = rd.pivot_table(index=["id", "tipo"], columns="modelo", values="cer").round(3)
tabla_cer

tesseract 5.5.2


,modelo,PaddleOCR 3.7.0 Parte 1,PaddleOCR 3.7.0 base,"Tesseract 5.5.2 (spa, psm 3)"
id,tipo,,,
doc_001,impreso,0.022,0.043,0.022
doc_004,impreso,0.258,0.083,0.689
doc_009,foto,0.542,0.000,1.000
doc_010,foto,0.162,0.398,0.427
doc_011,foto,0.500,0.929,0.929
doc_013,foto,0.251,0.282,0.390
doc_015,foto,0.091,0.750,1.000
doc_017,manuscrito,0.420,0.392,0.580
doc_018,manuscrito,0.615,0.623,0.954


## 4 · Audio: línea base y alternativas

| Modelo | Configuración (fijada antes de medir) |
|---|---|
| **faster-whisper small (base)** | `WhisperModel("small", device="cpu", compute_type="int8")`, `language="es"`, `beam_size=5`, `vad_filter=True` |
| **faster-whisper large-v3-turbo** | el modelo de la Parte 1 (`voz.Transcriptor("faster-whisper")`, int8 en CPU, VAD) |
| **Qwen3-ASR 0.6B** | `voz.Transcriptor("qwen3-asr-0.6b")` en la GPU de la Mac (MPS), VAD Silero para cortar tramos |

Se mide el **factor de tiempo real** (RTF = segundos de proceso / segundos de audio; < 1 es más rápido que tiempo real).
La carga del modelo no se cuenta y se hace una pasada de calentamiento antes de cronometrar. Todos sin diarización.

In [7]:
from faster_whisper import WhisperModel

audios = {c.id: voz.cargar_audio(c.archivo_audio) for c in ev_clips.itertuples()}
calentar = audios[EVAL_CLIPS[0]][: 5 * voz.SR]

def correr_asr(nombre, transcribir):
    transcribir(calentar)
    for c in ev_clips.itertuples():
        t0 = time.perf_counter()
        texto = transcribir(audios[c.id])
        seg = time.perf_counter() - t0
        texto = anon(texto)
        resultados_asr.append({"modelo": nombre, "id": c.id, "fuente_id": c.fuente_id, "tipo_audio": c.tipo_audio,
                               "texto": texto, "wer": wer_caso(c.texto_referencia, texto),
                               "segundos": seg, "rtf": seg / c.duracion_seg})

small = WhisperModel("small", device="cpu", compute_type="int8", cpu_threads=min(8, os.cpu_count() or 4))
def whisper_small(a):
    segs, _ = small.transcribe(a, language="es", beam_size=5, vad_filter=True)
    return " ".join(s.text.strip() for s in segs)
correr_asr("faster-whisper small (base, CPU)", whisper_small)
del small

In [8]:
turbo = voz.Transcriptor("faster-whisper")
correr_asr("faster-whisper large-v3-turbo (CPU)", lambda a: " ".join(s.texto for s in turbo.segmentos(a, idioma="es")))
del turbo

In [9]:
try:
    qwen = voz.Transcriptor("qwen3-asr-0.6b")
    correr_asr(f"Qwen3-ASR 0.6B ({qwen.device.upper()})", lambda a: " ".join(s.texto for s in qwen.segmentos(a, idioma="es")))
    del qwen
except Exception as e:   # si esta versión de transformers no trae Qwen3-ASR, se usa Parakeet (también de la Sesión 13)
    print(f"Qwen3-ASR no disponible ({type(e).__name__}: {str(e)[:120]}); se usa Parakeet TDT 0.6B v3")
    par = voz.Transcriptor("parakeet-v3")
    correr_asr(f"Parakeet TDT 0.6B v3 ({par.device.upper()})", lambda a: " ".join(s.texto for s in par.segmentos(a)))
    del par

ra = pd.DataFrame(resultados_asr)
tabla_wer = ra.pivot_table(index=["id", "tipo_audio"], columns="modelo", values="wer").round(3)
tabla_wer

Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

,modelo,Qwen3-ASR 0.6B (MPS),faster-whisper large-v3-turbo (CPU),"faster-whisper small (base, CPU)"
id,tipo_audio,,,
clip_004,entrevista,0.176,0.157,0.147
clip_013,entrevista,0.107,0.078,0.126
clip_024,entrevista,0.260,0.173,0.315
clip_030,entrevista,0.173,0.091,0.155
clip_036,entrevista,0.347,0.253,0.295
clip_042,entrevista,0.145,0.058,0.087
clip_052,noticiario,0.090,0.013,0.026
clip_056,noticiario,0.110,0.064,0.092
clip_058,habla leída,0.000,0.000,0.000


## 5 · Tabla comparativa (mismas 10 + 10 muestras)

In [10]:
def agregado(df, ref_col, metrica):
    filas = []
    for modelo, g in df.groupby("modelo", sort=False):
        ref = g["id"].map(ref_col)
        if metrica == "CER":
            valor = jiwer.cer([voz.normalizar(r) for r in ref], [voz.normalizar(h) or "∅" for h in g["texto"]])
            lat = f"{g['segundos'].mean():.2f} s/doc"
        else:
            valor = jiwer.wer([voz.normalizar(r) for r in ref], [voz.normalizar(limpiar_asr(h)) or "∅" for h in g["texto"]])
            lat = f"RTF {g['segundos'].sum() / ev_clips['duracion_seg'].sum():.2f}"
        filas.append({"modelo": modelo, "metrica": metrica, "valor": round(valor, 3), "latencia": lat})
    return filas

ref_docs = dict(zip(ev_docs["id"], ev_docs["texto_referencia"]))
ref_clips = dict(zip(ev_clips["id"], ev_clips["texto_referencia"]))
comparativa = pd.DataFrame(agregado(rd, ref_docs, "CER") + agregado(ra, ref_clips, "WER"))
comparativa["rol"] = ["línea base", "mejorado (Parte 1)", "alternativa", "línea base", "mejorado (Parte 1)", "alternativa"]
comparativa["costo"] = ["CPU · modelos PP-OCR (descarga única)", "CPU · mismos modelos + clasificador de orientación por línea",
                        "CPU · motor clásico LSTM, sin GPU ni descarga grande",
                        "CPU · 244 M parámetros (int8)", "CPU · 809 M parámetros (int8)", "GPU de la Mac (MPS) · 0,6 B parámetros"]
comparativa.to_csv(DATOS / "metricas_modelos.csv", index=False)
comparativa

,modelo,metrica,valor,latencia,rol,costo
0,PaddleOCR 3.7.0 base,CER,0.240,5.28 s/doc,línea base,CPU · modelos PP-OCR (descarga única)
1,PaddleOCR 3.7.0 Parte 1,CER,0.190,5.38 s/doc,mejorado (Parte 1),CPU · mismos modelos + clasificador de orienta...
2,"Tesseract 5.5.2 (spa, psm 3)",CER,0.362,0.38 s/doc,alternativa,"CPU · motor clásico LSTM, sin GPU ni descarga ..."
3,"faster-whisper small (base, CPU)",WER,0.160,RTF 0.11,línea base,CPU · 244 M parámetros (int8)
4,faster-whisper large-v3-turbo (CPU),WER,0.112,RTF 0.21,mejorado (Parte 1),CPU · 809 M parámetros (int8)
5,Qwen3-ASR 0.6B (MPS),WER,0.175,RTF 0.10,alternativa,"GPU de la Mac (MPS) · 0,6 B parámetros"


In [11]:
fig = px.bar(pd.concat([rd.assign(metrica="CER", error=rd["cer"], caso=rd["tipo"]),
                        ra.assign(metrica="WER", error=ra["wer"], caso=ra["tipo_audio"])]),
             x="caso", y="error", color="modelo", barmode="group", facet_col="metrica",
             title="Los manuscritos y los noticiarios con música concentran los errores en todos los modelos",
             labels={"error": "Tasa de error (media por caso)", "caso": ""})
fig.update_xaxes(matches=None); fig.update_yaxes(matches=None)
fig.show()

## 6 · Casos difíciles

- **Manuscritos:** CER por tipo de documento.
- **Audio con ruido o música:** WER por tipo de audio; además se cuentan palabras que el modelo produce de más
  (inserciones), que es como se ven las alucinaciones de Whisper sobre la música.

In [12]:
por_tipo_doc = rd.groupby(["tipo", "modelo"])["cer"].mean().unstack().round(3)
display(por_tipo_doc)

def inserciones(ref, hip):
    o = jiwer.process_words(voz.normalizar(ref), voz.normalizar(limpiar_asr(hip)) or "∅")
    return o.insertions
ra["inserciones"] = [inserciones(ref_clips[i], h) for i, h in zip(ra["id"], ra["texto"])]
por_tipo_aud = ra.groupby(["tipo_audio", "modelo"]).agg(wer=("wer", "mean"), inserciones=("inserciones", "sum")).round(3)
display(por_tipo_aud)

rd.to_csv(DATOS / "metricas_ocr_por_caso.csv", index=False)
ra.to_csv(DATOS / "metricas_asr_por_caso.csv", index=False)

modelo,PaddleOCR 3.7.0 Parte 1,PaddleOCR 3.7.0 base,"Tesseract 5.5.2 (spa, psm 3)"
tipo,,,
foto,0.309,0.472,0.749
impreso,0.140,0.063,0.356
manuscrito,0.547,0.383,0.835


wer  inserciones
tipo_audio  modelo                                                 
entrevista  Qwen3-ASR 0.6B (MPS)                 0.201           27
            faster-whisper large-v3-turbo (CPU)  0.135           20
            faster-whisper small (base, CPU)     0.187           28
habla leída Qwen3-ASR 0.6B (MPS)                 0.000            0
            faster-whisper large-v3-turbo (CPU)  0.000            0
            faster-whisper small (base, CPU)     0.000            0
noticiario  Qwen3-ASR 0.6B (MPS)                 0.100            2
            faster-whisper large-v3-turbo (CPU)  0.039            0
            faster-whisper small (base, CPU)     0.059            0

In [13]:
# Ejemplo concreto: la referencia frente a cada modelo en el noticiario con música
caso = ev_clips[ev_clips["tipo_audio"] == "noticiario"]["id"].iloc[0]
print("REFERENCIA:\n", ref_clips[caso], "\n")
for r in ra[ra["id"] == caso].itertuples():
    print(f"{r.modelo} (WER {r.wer:.2f}):\n", r.texto, "\n")

REFERENCIA:
 Rincones romáticos de la Lima Vieja
evocación maravillosa del pasado
acá está la iglesia donde parece que el tiempo se hubiera detenido como en un remanso de paz
su reja labrada y sus muros blancos ante los que el oleaje de la vida muere suavemente como el mar en las arenas de la playa
Aquí está la iglesia del patrocinio que nos dice de los días ingenuos de la colonia de fe inconmovible y de emoción de santidad 

faster-whisper small (base, CPU) (WER 0.03):
 Rincones románticos de la lima vieja, evocación maravillosa del pasado. Acá está la iglesia donde parece que el tiempo se hubiera detenido, como en un remanso de paz. Su reja labrada y sus muros blancos, ante los que el oleaje de la vida muere suavemente, como el mar en las arenas de la playa. Aquí está la iglesia del patrocinio que nos dice de los días ingenios de la colonia, de fe inconmovible y de emoción de santidad. 

faster-whisper large-v3-turbo (CPU) (WER 0.01):
 Rincones románticos de la Lima Vieja Evocación m

## 7 · Extracción estructurada con un LLM

Gemini (API, `temperature=0`, respuesta forzada a JSON) lee el texto OCR/ASR **ya seudonimizado** de la Parte 1 y devuelve
`modalidad`, `tema`, `resumen`, `emisor`, `fecha`, `monto`, `lugar`, `citas_clave` y `numero_hablantes`, con `null` cuando el
dato no está en el texto. El prompt completo está en `llm.py`. Las frases de OpenSLR 73 (una oración leída cada una) no se envían: no tienen contenido que extraer.

In [14]:
import importlib; importlib.reload(llm)
MODELO_LLM = llm.elegir_modelo()
print("Modelo principal:", MODELO_LLM)

entradas = pd.concat([
    docs.assign(modalidad="documento", texto=docs["texto_ocr"])[["id", "modalidad", "texto"]],
    clips[clips["fuente_id"] != "openslr73"].assign(modalidad="audio", texto=clips["texto_asr"])[["id", "modalidad", "texto"]],
], ignore_index=True)

filas = []
for e in entradas.itertuples():
    campos = llm.extraer(e.id, e.modalidad, e.texto, MODELO_LLM)
    usado = campos.pop("_modelo", MODELO_LLM)          # si el modelo principal estaba saturado se usó otro
    filas.append({"id": e.id, "modalidad": e.modalidad, "campos_json": json.dumps(campos, ensure_ascii=False),
                  "modelo_llm": usado, "fecha": date.today().isoformat()})
extraccion = pd.DataFrame(filas)
extraccion.to_csv(DATOS / "extraccion_llm.csv", index=False)
print(extraccion.shape, extraccion["modelo_llm"].value_counts().to_dict())
extraccion.head()

  … gemini-3-flash-preview respondió 429; reintento en 15 s
  gemini-3-flash-preview no disponible: HTTP Error 429: Too Many Requests
  … gemini-3.1-flash-lite respondió 503; reintento en 15 s
Modelo principal: gemini-3.1-flash-lite
(82, 5) {'gemini-3.1-flash-lite': 65, 'gemini-3-flash-preview': 17}


,id,modalidad,campos_json,modelo_llm,fecha
0,doc_001,documento,"{""modalidad"": ""documento"", ""tema"": ""Reconocimi...",gemini-3-flash-preview,2026-10-26
1,doc_002,documento,"{""modalidad"": ""documento"", ""tema"": ""Diversione...",gemini-3-flash-preview,2026-10-26
2,doc_003,documento,"{""modalidad"": ""documento"", ""tema"": ""Idea gener...",gemini-3-flash-preview,2026-10-26
3,doc_004,documento,"{""modalidad"": ""documento"", ""tema"": ""Constituci...",gemini-3-flash-preview,2026-10-26
4,doc_005,documento,"{""modalidad"": ""documento"", ""tema"": ""Prospecto ...",gemini-3-flash-preview,2026-10-26


In [15]:
# Chequeo automático de "no inventar" (complementa, no reemplaza, la auditoría humana):
# ¿las citas aparecen literalmente en el texto? ¿la fecha y el monto aparecen en el texto?
texto_de = dict(zip(entradas["id"], entradas["texto"]))
def soporte(fila):
    c, t = json.loads(fila.campos_json), voz.normalizar(texto_de[fila.id])
    citas = c.get("citas_clave") or []
    ok_citas = all(voz.normalizar(q) in t for q in citas)
    ok_fecha = c.get("fecha") in (None, "") or any(x in t for x in voz.normalizar(str(c["fecha"])).split() if x.isdigit())
    return pd.Series({"citas_literales": ok_citas, "fecha_en_texto": ok_fecha})
chequeo = extraccion.join(extraccion.apply(soporte, axis=1))
chequeo[["citas_literales", "fecha_en_texto"]].mean().round(3).rename("proporción OK").to_frame()

,proporción OK
citas_literales,0.988
fecha_en_texto,0.988


### Auditoría humana (20 extracciones al azar: 10 documentos y 10 audios)

**Criterio de corrección** (fijado antes de revisar): una extracción es **completamente correcta** si

1. `tema` y `resumen` describen fielmente el texto (no agregan hechos que no están);
2. cada campo no nulo (`emisor`, `fecha`, `monto`, `lugar`) aparece en el texto y está bien asignado;
3. no deja en `null` un dato que está claramente en el texto;
4. las `citas_clave` son literales y `numero_hablantes` coincide con las etiquetas.

Un solo error hace que la extracción cuente como incorrecta. La revisión se anotó en `datos/auditoria_llm.csv`
(columna `correcto` = 1/0 y `error` con la descripción).

In [16]:
ruta_aud = DATOS / "auditoria_llm.csv"
if not ruta_aud.exists():
    muestra = pd.concat([extraccion[extraccion["modalidad"] == "documento"].sample(10, random_state=42),
                         extraccion[extraccion["modalidad"] == "audio"].sample(10, random_state=42)])
    muestra = muestra.assign(texto=muestra["id"].map(texto_de).str[:1500], correcto="", error="")
    muestra[["id", "modalidad", "texto", "campos_json", "correcto", "error"]].to_csv(ruta_aud, index=False)
    print("Plantilla creada en datos/auditoria_llm.csv: complétala y vuelve a correr esta celda.")

auditoria = pd.read_csv(ruta_aud)
assert auditoria["correcto"].notna().all(), "Faltan filas por auditar en datos/auditoria_llm.csv"
correctas = int(auditoria["correcto"].sum())
print(f"Extracciones completamente correctas: {correctas}/20 = {correctas / 20:.0%}")
print(auditoria.groupby("modalidad")["correcto"].agg(["sum", "count"]))
auditoria.loc[auditoria["correcto"] == 0, ["id", "modalidad", "error"]]

Extracciones completamente correctas: 17/20 = 85%
           sum  count
modalidad            
audio       10     10
documento    7     10


,id,modalidad,error
4,doc_012,documento,fecha «14 FEB» armada a partir de fragmentos s...
7,doc_002,documento,lugar «Lima» no aparece en el texto (completad...
9,doc_006,documento,emisor «Los editores» y lugar «Lima» no aparec...


## 8 · Decisión y reflexión

**Documentos.** La configuración de la Parte 1 (PaddleOCR con orientación por línea) obtiene el mejor CER agregado
(**0,19**, frente a 0,24 de la base y 0,36 de Tesseract). Pero el CER agregado pesa por caracteres y lo domina el documento
más largo (la resolución de Mache, CER 0,02). Por tipo, ninguna configuración gana en todo: la Parte 1 mejora mucho las
**fotos** (CER medio 0,47 → 0,31; la placa de mármol del mercado pasa de 0,75 a 0,09), pero la configuración base es mejor
en **impresos** (0,06 frente a 0,14) y **manuscritos** (0,38 frente a 0,55): la orientación por línea a veces gira mal
líneas que ya estaban derechas (Calle Capón, encabezados de Guaman Poma). Tesseract es 14 veces más rápido (0,4 s frente
a 5,4 s por documento) y empata en el impreso moderno, pero se derrumba en fotos (0,75) y manuscritos (0,84).

**Audio.** faster-whisper **large-v3-turbo** es el más preciso (WER **0,11**), con ~30 % menos errores que la línea base
*small* (0,16) y que Qwen3-ASR 0.6B (0,18). Es la mitad de rápido que ellos (RTF 0,21 frente a 0,10–0,11), pero aun así
transcribe una hora de audio en ~13 minutos en CPU.

**Por tipo de documento.** Mejor desempeño: impresos tipográficos limpios (casi sin errores). Peor: manuscritos
(CER 0,38–0,55 en el mejor caso), porque los modelos PP-OCR están entrenados sobre todo con texto impreso; la cursiva de
Palma, el recibo de 1895 llenado a mano y la letra del siglo XVII de Guaman Poma, con ortografía antigua y palabras en
quechua, quedan fuera de esa distribución. También fallan las letras decorativas: el cartel pintado a mano «Picante de cuy»
tiene CER 0,93 con la base y Tesseract.

**Por tipo de audio.** Contra lo esperado, los **noticiarios con música** fueron fáciles (WER 0,04 con turbo): el locutor
profesional habla claro y la música queda debajo de la voz, y el VAD evitó alucinaciones. El **habla leída** de OpenSLR no
tuvo errores. Lo más difícil fueron las **entrevistas en la calle** (WER 0,14 con turbo y hasta 0,25 en un clip de Cusco):
ruido de tráfico y multitud, voces que se superponen y habla espontánea con muletillas y frases cortadas.

**Extracción con LLM.** 17/20 extracciones auditadas fueron completamente correctas (85 %). Los 3 errores tienen el mismo
patrón: el modelo **completó con conocimiento externo** datos que el texto no dice (lugar «Lima», emisor «Los editores»,
fecha «14 FEB» con tema «San Valentín»), a pesar de que el prompt lo prohíbe, y siempre en documentos con OCR muy ruidoso.

**Elección para un entorno real.**
- **OCR:** PaddleOCR, eligiendo la configuración según el soporte: orientación por línea para fotos y desactivada para
  impresos y manuscritos. El costo (5 s por página en CPU) es aceptable para un archivo; Tesseract solo como filtro
  rápido para impresos limpios. Los manuscritos necesitan revisión humana o un modelo de escritura a mano.
- **ASR:** faster-whisper large-v3-turbo en CPU, por su precisión y porque no requiere GPU; *small* si se necesita el
  doble de velocidad, aceptando ~45 % más errores.
- **LLM:** útil para estructurar, pero con una regla automática que descarte todo campo que no aparezca en el texto
  fuente y con auditoría humana de una muestra.

In [17]:
mejor_ocr = comparativa[comparativa["metrica"] == "CER"].sort_values("valor").iloc[0]
mejor_asr = comparativa[comparativa["metrica"] == "WER"].sort_values("valor").iloc[0]
print(f"OCR elegido: {mejor_ocr.modelo} · ASR elegido: {mejor_asr.modelo}")
print(f"Resultados Test: WER: {mejor_asr.valor:.2f} | CER: {mejor_ocr.valor:.2f} | Acuerdo LLM-Humano: {correctas}/20")

OCR elegido: PaddleOCR 3.7.0 Parte 1 · ASR elegido: faster-whisper large-v3-turbo (CPU)
Resultados Test: WER: 0.11 | CER: 0.19 | Acuerdo LLM-Humano: 17/20
